# Lab 1: Brilliant by Chance

*Financial Engineering in Practice*, Chapter 1.

**What you will explore:** how good a track record luck alone can produce, and how high the bar must be before a record counts as evidence of skill.

You will build 1,000 market-timing strategies that switch between stocks and Treasury bills **at random**. Every one has zero skill by construction. You will backtest them on the first half of nearly a century of U.S. market data, pick the winners, and see what happens to them afterward.

This notebook walks through the lab step by step. `python lab01.py` runs the same analysis in one go and writes a full report.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import lab01

SOURCE = "french"   # "french" downloads real data; "sim" uses fake data to test the code
DATA_FILE = None    # or the path of a French library zip you downloaded yourself
OUT = Path("report")
OUT.mkdir(exist_ok=True)

mkt = lab01.load_market(SOURCE, DATA_FILE)
SIMULATED = mkt.attrs.get("simulated", False)
print(f"{len(mkt)} months, {mkt.index[0]} to {mkt.index[-1]}", "(SIMULATED)" if SIMULATED else "")
mkt.tail()

## Part A: Signal and noise

How good is the stock market itself, by the Sharpe ratio, and how much does a 10-year record vary?

In [ ]:
a = lab01.part_a(mkt, window_years=10)
print(f"Market excess return {a['mean_excess_ann']:.1%} a year, volatility {a['vol_ann']:.1%}")
print(f"Sharpe ratio {a['sharpe']:.2f}, t = {a['t_stat']:.2f} over {a['years']:.0f} years")
print(f"10-year Sharpe ratios ranged from {a['rolling_min']:.2f} to {a['rolling_max']:.2f}")
lab01.fig_rolling(a, OUT, SIMULATED)
display(Image(OUT / "fig1_1_rolling_sharpe.png"))
a["detect"]

## Part B: 1,000 strategies with zero skill

**Predict before you run.** Out of 1,000 strategies with no skill, how many do you expect to pass the usual significance test (t > 1.96) in the backtest? How large will the best t-statistic be? Write your guesses down.

In [ ]:
b = lab01.part_b(mkt, n_strategies=1000, seed=2026)
s = lab01.summarize_b(b)
print(f"Backtest: {s['in_sample']['start']} to {s['in_sample']['end']}")
print(f"{s['count_t_above_1_96']} strategies pass t > 1.96 (about {s['expected_count_t_above_1_96']:.0f} expected by luck)")
print(f"Best backtest: alpha {s['best_alpha_in_sample_pct']:+.2f}% a year, t = {s['best_t_in_sample']:.2f}")
lab01.fig_histogram(b, s, OUT, SIMULATED)
display(Image(OUT / "fig1_2_t_histogram.png"))

Now follow the winners into the second half of the sample, the period an investor who picked them would actually have lived through.

In [ ]:
print(f"After selection: {s['out_of_sample']['start']} to {s['out_of_sample']['end']}")
print(f"Best strategy afterward: alpha {s['best_alpha_out_of_sample_pct']:+.2f}% a year, t = {s['best_t_out_of_sample']:.2f}")
print(f"Correlation between backtest t and later t: {s['correlation_t_in_vs_out']:+.2f}")
lab01.fig_best(mkt, b, s, OUT, SIMULATED)
lab01.fig_scatter(b, s, OUT, SIMULATED)
display(Image(OUT / "fig1_3_best_strategy.png"))
display(Image(OUT / "fig1_4_in_vs_out.png"))
s["top10"].round(2)

## Part C: The cost of searching

How does the best backtest improve as you try more strategies?

In [ ]:
c = lab01.part_c(b)
lab01.fig_search(c, OUT, SIMULATED)
display(Image(OUT / "fig1_5_search_cost.png"))
c.round(2)

## Questions to think about

1. Does an impressive backtest, on its own, show skill? Answer in two or three sentences using your numbers.
2. Name one piece of information you would ask for before believing a backtest.
3. Change `seed` in Part B and re-run. How much does the number of "significant" strategies change, and why might it differ from the 25 expected?
4. Set `n_strategies=10000`. What happens to the best backtest t-statistic?